# SmartFit try-on baseline: IDM-VTON (spec §8.1)

Runs the pretrained IDM-VTON model, **no training**, on the SmartFit data pack, and records
time per image, peak GPU memory and paired SSIM/LPIPS. Run the other model's notebook too, then compare
both with `compare.py`.

**Before you click Run All** (right-hand panel → Session options):
1. **Accelerator:** **GPU T4 x2** (needed: the weights are ~14 GB, so the garment UNet runs on the second T4; on a single GPU it falls back to much slower CPU offload).
2. **Internet:** On. It downloads the code, the Python packages and about 28 GB (two SDXL-size UNets, text and image encoders).
3. **Input → Add Input:** your *private* SmartFit try-on pack dataset.
4. Optional: **Add-ons → Secrets:** `HF_TOKEN` (a Hugging Face *read* token). The models are public,
   so this only avoids anonymous rate limits.

Expected run time: roughly 1-1.5 hours (about 10 min of downloads and conversion, then the 36 images).

In [ ]:
import glob, json, os

hits = sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
assert hits, "No manifest.json found: add the SmartFit try-on pack dataset as an input."
PACK = os.path.dirname(hits[0])
manifest = json.load(open(f"{PACK}/manifest.json"))
print("pack:", PACK, "|", len(manifest["jobs"]), "jobs")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("No HF_TOKEN secret: anonymous downloads (fine for these public models)")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code at a fixed commit and an isolated environment

Safe to re-run in the same session: the checkout, the env and downloaded weights in `/tmp` are reused.

In [ ]:
![ -d /tmp/IDM-VTON ] || git clone -q https://github.com/yisol/IDM-VTON.git /tmp/IDM-VTON
!git -C /tmp/IDM-VTON checkout -q 0d5f3ec2d737487a9bb24e4100936ad254780383 && git -C /tmp/IDM-VTON log -1 --format="%H %cd"
!pip install -q uv==0.5.11
![ -x /tmp/env/bin/python ] || uv venv -q --python 3.11 /tmp/env
!uv pip install -q --python /tmp/env/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /tmp/env/bin/python -r {PACK}/code/requirements-idmvton.txt
!/tmp/env/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Run every job in the pack
One line per job: `ok`, `oom` or `error`. A failed job does not stop the run.

In [ ]:
!/tmp/env/bin/python {PACK}/code/run_idmvton.py --pack {PACK} --repo /tmp/IDM-VTON --out /kaggle/working/results/idmvton --placement auto

## 3. Summary and a few results

In [ ]:
from IPython.display import Image as Show, display

r = json.load(open("/kaggle/working/results/idmvton/results.json"))
print(json.dumps({k: r[k] for k in ("settings", "summary", "metrics", "environment")}, indent=2))
for row in [j for j in r["jobs"] if j["status"] == "ok"][:4]:
    print(row["id"], row.get("seconds"), "s", "ssim", row.get("ssim"), "lpips", row.get("lpips"))
    display(Show(filename="/kaggle/working/results/idmvton/" + row["image"], width=288))

## 4. Download

Zips the results. Download `results_idmvton.zip` from the **Output** panel (or after *Save Version*), unzip it next
to your local pack folder and run `compare.py` (see `backend/services/tryon/baseline/README.md`).

In [ ]:
!cd /kaggle/working && rm -f results_idmvton.zip && zip -qr results_idmvton.zip results/idmvton && ls -la /kaggle/working